<img src="../logo.png" alt="Vegeta" width="240">

# prop_lab — Kestrel's 9×6 propeller alone in CFD, and what it hands to the aircraft

Everything Kestrel says about its propellers rests on one assumption: Boreas's generic thin-airfoil section for a
9-inch blade. The drive tables (full thrust, torque-to-thrust, power against the inflow) are its blade-element
numbers; the propeller normal force in the transition is De Young's estimate; the rotor disks in the airframe CFD
are blade-element sources fed with the same blade rows and polar. **This notebook measures the propeller itself**:
the 9×6 resolved — blades and hub — in a rotating frame, in the configurations the aircraft flies:

| configuration | template | what Kestrel reads from it |
|---|---|---|
| the hover: static thrust at three speeds (the hover rpm among them) | `rotor_mrf_static` | thrust and power per rpm — the drive table's first row, the figure of merit |
| axial inflow 4–24 m/s at 8000 rpm, two points at 10 000, the cruise at its own rpm | `rotor_mrf` | thrust, torque, efficiency against the advance ratio — the drive tables and `rpm_for` out to the pitch speed |
| inflow at 15–90° to the axis at 8 and 14 m/s, the blades frozen at 0° and at 90° | `rotor_mrf_oblique` (new) | the **normal force** (the mean over the two azimuths) and the propeller's own pitching moment — the transition model's `prop_cn_alpha`; the thrust loss off-axis |

Each case is read beside blade-element theory at the same point. What comes out is a **calibration** — two scales
for the blade section (lift slope, profile drag) that make BEMT match the solved hover and axial cases in thrust and
torque together, and one factor on the normal-force slope — written with the whole table to
`designs/data/prop_lab.json`. `kestrel_systems.propulsion` applies the section scales (the drive tables, `rpm_for`
and the rotor disks' polar tables then carry the measured propeller), `kestrel_flight.prop_cn_alpha` the normal-force
factor; notebooks 39 and 39 say on every run whether the calibration is on file. The last section turns the same
loads into **noise** — the blade-passing frequency and its harmonics, the broadband, dB(A) at a bystander — and
renders the sound as WAV files (one rotor, the four rotors beating in the hover, the transition as a glide).

**A failure is a result.** A case that fails in a step or runs into nonsense is listed as such and left out of the
fit. Without OpenFOAM (this copy) every case is NOT RUN and the calibration stays at unity. **A calibration must earn
its place**: it is applied only when the fit actually fits (rms of the log ratios ≤ 0.15), the section scales stay
within 0.7–1.3 (further and the section's *shape* — or the mesh — is wrong, not its size), and the normal-force factor
comes from oblique cases run at **two blade azimuths** (a frozen rotor at one azimuth sees the cross-flow symmetrically
and reports no normal force) and lies within 0.3–3. What fails the gates is written down with its reason and the
drives stay on Boreas.

| variable | default | meaning |
|---|---|---|
| `VEGETA_SKIP_OPENFOAM=1` | unset | list and prepare only |
| `PROP_LAB_QUALITY` | `screening` | `screening` (10 cells per diameter, surface level 4, 600 iterations: ~0.5–1 M cells) or `fine` (14 / 5 / 1000). The first campaign ran 6 cells per diameter at level 3 — 100 k cells, four cells across the chord — and gave a figure of merit of 0.29: unusable |
| `PROP_LAB_JOBS` / `PROP_LAB_PROCESSORS` | 1 / 4 | cases in parallel / MPI ranks per case |
| `PROP_LAB_ONLY` | unset | `hover`, `axial`, `oblique` or a comma list |

In [ ]:
# DEBUG — ON BY DEFAULT: the expensive solvers are mocked (vegeta.mock: fixed numbers, seconds instead of hours,
# labelled MOCK). For real results set DEBUG = False here, or from the shell: VEGETA_DEBUG=0 jupyter nbconvert …
DEBUG = True
from vegeta import mock
DEBUG = mock.setup(DEBUG)

In [ ]:
import json, math, os, sys, datetime
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from vegeta import dedalus, aeromant, boreas
sys.path.insert(0, "../../designs")
import prop_lab as pl, kestrel_systems as ks, kestrel_flight as kf, kestrel_loop as kl

pd.set_option("display.max_colwidth", 70); pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)
RUNS = Path("../_runs/prop_lab"); RUNS.mkdir(parents=True, exist_ok=True)
OUT = Path("../output/38_prop_lab"); OUT.mkdir(parents=True, exist_ok=True)
RUN_CFD = os.environ.get("VEGETA_SKIP_OPENFOAM") != "1"
QUALITY = os.environ.get("PROP_LAB_QUALITY", "screening")
JOBS, PROCS = int(os.environ.get("PROP_LAB_JOBS", "1")), int(os.environ.get("PROP_LAB_PROCESSORS", "4"))
ONLY = [s.strip() for s in os.environ.get("PROP_LAB_ONLY", "hover,axial,oblique").split(",") if s.strip()]
TODAY = datetime.date.today().isoformat()
print(f"CFD {'on' if RUN_CFD else 'NOT RUN (VEGETA_SKIP_OPENFOAM=1)'} | quality {QUALITY} {pl.QUALITY[QUALITY]} | jobs {JOBS} × {PROCS} ranks | kinds {ONLY}")
CAL0 = ks.prop_lab_calibration()
print("calibration on file before this run:", CAL0 if CAL0["on_file"] else "none (unity scales)")
from vegeta import yanus
GATE = yanus.enter("38_prop_lab")                   # the door keeper (workflows/flow.py): this notebook's needs; what it gives is checked at the end


## 1. The propeller

The 9×6 as Boreas carries it (`kestrel_systems.COMPONENTS`: constant geometric pitch 152 mm, chord 14 → 21 → 8 mm,
two blades) built by Dedalus's parametric propeller — NACA-4 sections 10 % thick with 4 % camber, a 16 × 10 mm hub —
and turned onto the rotor frame's axis (+x). The real APC blade's planform and twist are not in the model: this is
the propeller Kestrel's numbers assume, measured as such.

In [ ]:
G = pl.geometry()
display(pd.Series(G, name="the 9x6 (mm, -)").to_frame().T)
CAD = pl.cad()
display(pd.Series({k: (round(v, 1) if isinstance(v, float) else v) for k, v in CAD.measure().items() if k in ("volume", "surface_area", "dimensions")}, name="CAD").to_frame().T)
dedalus.plot_views(CAD, figsize=(12, 4)); plt.show()
STL = pl.stl(RUNS / "stl")
print("STL in the rotor frame (axis +x):", STL)
PROP = ks.propulsion().prop
BLADE = pd.DataFrame({"r [m]": PROP.r, "twist [deg]": PROP.beta_deg, "chord [m]": PROP.chord})
display(BLADE.T.round(4))

## 2. The blade-element baseline

What Boreas predicts at every configuration before any solver runs: thrust, torque, shaft power, efficiency (figure
of merit in the hover), the advance ratio — and for the oblique cases De Young's normal-force slope, the number the
transition model uses. The hover rpm (one rotor's share of the 1.69 kg aircraft) and the cruise rpm (0.8 N at
16 m/s) come from the same model.

In [ ]:
SPECS = {k: v for k, v in pl.configurations().items() if v["kind"] in ONLY}
T0 = pl.table(None, SPECS)
display(T0.drop(columns=["status", "note"]).astype({c: float for c in T0.columns if c not in ("kind", "template", "status", "note")}).round(4))
print(f"hover rpm {pl.rpm_hover():.0f}, cruise rpm {pl.rpm_cruise():.0f}; {len(SPECS)} cases")
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
rpms = np.linspace(4000, 11000, 15)
for V in (0.0, 8.0, 16.0, 24.0):
    ops = [pl.bemt(r, V) for r in rpms]
    ax[0].plot(rpms, [o.thrust for o in ops], label=f"{V:g} m/s"); ax[1].plot(rpms, [o.power for o in ops], label=f"{V:g} m/s")
ax[0].set(xlabel="rpm", ylabel="N", title="BEMT thrust"); ax[1].set(xlabel="rpm", ylabel="W", title="BEMT shaft power"); ax[0].legend(fontsize=8)
Js = np.linspace(0.05, 0.9, 30)
n = 8000 / 60
ops = [pl.bemt(8000.0, j * n * PROP.diameter) for j in Js]
ax[2].plot(Js, [o.ct for o in ops], label="CT"); ax[2].plot(Js, [o.cp for o in ops], label="CP"); ax[2].plot(Js, [o.efficiency for o in ops], label="η")
ax[2].set(xlabel="J = V / (n D)", title="BEMT at 8000 rpm"); ax[2].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 3. The campaign

Every case prepared (the STL, the MRF zone, the mesh and solver settings of the chosen quality), run through the
cache, read beside its blade-element row. The table is the result whatever happened in the solver.

In [ ]:
TAB = T0
if RUN_CFD:
    try:
        CASES = pl.plan(RUNS, SPECS, quality=QUALITY)
        names = list(CASES)
        res = aeromant.run_cases([CASES[n] for n in names], jobs=JOBS, processors=PROCS, progress=True, cache=True)
        TAB = pl.table(dict(zip(names, res)), SPECS)
    except Exception as exc:                        # no OpenFOAM, no CAD kernel, a broken environment: the listing stays NOT RUN
        print(f"CFD not run — {type(exc).__name__}: {str(exc)[:200]}")
SM = pl.summary(TAB)
print(SM["line"])
cols = [c for c in TAB.columns if c not in ("template", "Re (0.75 R)")]
display(TAB[cols])
TAB.to_csv(OUT / "prop_lab_cases.csv")
fails = TAB[TAB["status"].isin(["FAILED", "diverged"])]
if len(fails):
    print(f"{len(fails)} cases did not solve:"); display(fails[["kind", "rpm", "V [m/s]", "angle [deg]", "status", "note"]])

## 4. CFD against blade-element theory

The hover and axial cases: thrust and torque ratios against the advance ratio; the oblique cases: the normal-force
slope against De Young's and the thrust's fall-off with the angle.

In [ ]:
SOL = TAB[TAB["status"] == "solved"]
if len(SOL):
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
    ax_ = SOL[SOL["kind"].isin(["hover", "axial"])]
    ax[0].plot(ax_["J"].astype(float), ax_["T CFD / BEMT"].astype(float), "o", label="thrust"); ax[0].plot(ax_["J"].astype(float), ax_["Q CFD / BEMT"].astype(float), "s", label="torque")
    ax[0].axhline(1, color="k", lw=0.8); ax[0].set(xlabel="J", ylabel="CFD / BEMT", title="hover and axial inflow"); ax[0].legend(fontsize=8)
    ob = SOL[SOL["kind"] == "oblique"]
    if len(ob):
        for V, g in ob.groupby("V [m/s]"):
            ax[1].plot(g["angle [deg]"].astype(float), g["CFD CNα [1/rad]"].astype(float), "o-", label=f"CFD, {V:g} m/s")
            ax[1].plot(g["angle [deg]"].astype(float), g["De Young CNα [1/rad]"].astype(float), "--", label=f"De Young, {V:g} m/s")
            ax[2].plot(g["angle [deg]"].astype(float), g["T CFD / BEMT"].astype(float), "o-", label=f"{V:g} m/s")
        ax[1].set(xlabel="inflow angle [deg]", ylabel="dCN/dα [1/rad]", title="the propeller's normal force"); ax[1].legend(fontsize=7)
        ax[2].set(xlabel="inflow angle [deg]", ylabel="T CFD / BEMT (axial component)", title="thrust off-axis"); ax[2].legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("nothing solved: the plots wait for a run with OpenFOAM")

## 5. The calibration, and what Kestrel reads

Two scales on the blade section — the lift slope and the profile drag — fitted so that blade-element theory
reproduces the solved hover and axial cases in thrust **and** torque (a least-squares fit on the log ratios), and the
median of the measured normal-force slope over De Young's. Exported to `designs/data/prop_lab.json`; then the drive
table rebuilt on the calibrated section against the one Kestrel used so far.

In [ ]:
SEC, NF = pl.fit_section(TAB), pl.fit_normal_force(TAB)
display(pd.Series({k: v for k, v in SEC.items()}, name="section fit").to_frame().T)
display(pd.Series({k: v for k, v in NF.items() if k != "points"}, name="normal-force fit").to_frame().T)
if NF["points"]:
    display(pd.DataFrame(NF["points"]).round(4))
ACC = pl.accept(SEC, NF)
print("applied:", ACC["applied"] or "nothing"); [print("rejected —", r) for r in ACC["rejected"]]
DOC = pl.export(TAB, pl.PROP_LAB_JSON, meta={"source": "38_prop_lab", "quality": QUALITY, "date": TODAY, "propeller": "9x6, Dedalus CAD of Boreas's planform"})
CAL = ks.prop_lab_calibration()
print("calibration on file now:", CAL)
ratios = TAB[(TAB["status"] == "solved") & TAB["kind"].isin(["hover", "axial"])]
drift = float(np.polyfit(ratios["J"].astype(float), np.log(ratios["T CFD / BEMT"].astype(float).clip(lower=1e-3)), 1)[0]) if len(ratios) >= 3 and "T CFD / BEMT" in ratios.columns else float("nan")
fm = float(TAB.loc[[i for i in TAB.index if "the hover" in i][0], "CFD η / FM"]) if any("the hover" in i for i in TAB.index) and "CFD η / FM" in TAB.columns else float("nan")
CHK = [kl.check("hover cases solved", SM["by kind"]["hover"], "≥", 2), kl.check("axial cases solved", SM["by kind"]["axial"], "≥", 5),
       kl.check("oblique cases solved (both azimuths)", SM["by kind"]["oblique"], "≥", 8),
       kl.check("hover figure of merit (CFD)", fm, "≥", 0.45, note="below this the blade is under-resolved: a finer mesh, not a calibration"),
       kl.check("section fit: rms log ratio after", SEC["rms_log_ratio_after"] if SEC["rms_log_ratio_after"] is not None else float("nan"), "≤", pl.GATES["rms_after"], note="thrust and torque follow one scaled section"),
       kl.check("thrust ratio's drift with J (d ln / dJ)", abs(drift), "≤", 1.0, note="a ratio that falls with J means the section's shape is wrong, not its scale"),
       kl.check("cl scale within the band", pl.GATES["cl_scale"][0] <= (SEC["cl_scale"] or 1.0) <= pl.GATES["cl_scale"][1], "==", True),
       kl.check("normal force from both azimuths", bool(NF.get("azimuths_averaged")), "==", True),
       kl.check("cn factor within the band", pl.GATES["cn_factor"][0] <= (NF["cn_factor"] or 1.0) <= pl.GATES["cn_factor"][1], "==", True),
       kl.check("calibration accepted and applied", ACC["accepted"], "==", True)]
display(kl.check_table(CHK, "the campaign"))
if CAL["on_file"]:
    DR = ks.drive_table(recompute=True, save=True)         # the drive table on the calibrated section (the key stamps the scales)
    print("drive table rebuilt on the calibrated section:", DR.source, "| T_max at 0 / 16 m/s:", round(DR.T_max[0], 2), round(float(DR.t_max(16.0)), 2), "N")
    hp = ks.hover_point(1.686)
    print(f"hover: {hp['rpm']:.0f} rpm, {hp['electrical_W']:.0f} W per drive, T/W {hp['T_W']:.2f}")
else:
    print("no calibration yet: Kestrel's drive tables and normal force stay on the generic section (unity scales)")

## 6. Noise: the frequencies, and what it sounds like

The **blade-passing frequency** is B·n — two blades at the hover's 6700 rpm give 224 Hz, the cruise's lower rpm a
lower note, 10 000 rpm 333 Hz — and the tonal noise of the steady blade loading (Gutin: the thrust and torque the CFD
measured, BEMT's where it did not) sits on it and its harmonics; the broadband vortex noise (Hubbard's estimate from
the blade area and the section speed at 0.7 R) peaks higher, around the Strouhal frequency of the blade's thickness.
Both at a bystander 10 m off, 60° from the axis, in dB and dB(A). A steady estimate: the once-per-revolution load cycle
of an inclined inflow adds loading harmonics it does not carry — the oblique cases' loads are the mean ones.

Then the sound itself, rendered: one rotor in the hover, in cruise, at a transition point; the **four rotors in the
hover** — the allocation holds the heading with a small split between the diagonal pairs, so the four blade-passing
tones beat against each other rather than hum; and the **transition** as a glide from the hover rpm through the
oblique points to the cruise. One full scale for every file: the louder case is louder on playback.

In [ ]:
from IPython.display import Audio
NZ = pl.noise_table(TAB)
display(NZ.astype({c: float for c in NZ.columns if c not in ("source",)}).round(1))
NZ.to_csv(OUT / "prop_lab_noise.csv")
r_h = pl.rpm_hover()
HOV = pl.noise(r_h, 0.0, pl.bemt(r_h).thrust, pl.bemt(r_h).torque)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
for nz, lab in ((HOV, f"hover, {r_h:.0f} rpm"), (pl.noise(pl.rpm_cruise(), 16.0, pl.bemt(pl.rpm_cruise(), 16.0).thrust, pl.bemt(pl.rpm_cruise(), 16.0).torque), f"cruise, {pl.rpm_cruise():.0f} rpm"),
                (pl.noise(10000.0, 0.0, pl.bemt(10000.0).thrust, pl.bemt(10000.0).torque), "10 000 rpm")):
    f = [t[0] for t in nz["tones"]]; L = [t[1] for t in nz["tones"]]
    ax[0].stem(f, L, label=f"{lab}: tones", basefmt=" "); ax[0].plot(nz["broadband_peak_hz"], nz["broadband_db"], "v", ms=9, label=f"{lab}: broadband peak")
ax[0].set(xlabel="Hz", ylabel="dB re 20 µPa at 10 m, 60°", title="the blade-passing tones and the broadband", xscale="log"); ax[0].legend(fontsize=7)
rows = NZ[NZ["angle [deg]"].astype(float) == 0.0]
ax[1].plot(rows["rpm"].astype(float), rows["total [dB(A)]"].astype(float), "o", label="total dB(A)"); ax[1].plot(rows["rpm"].astype(float), rows["tonal [dB]"].astype(float), "s", label="tonal dB")
ax[1].plot(rows["rpm"].astype(float), rows["broadband [dB]"].astype(float), "^", label="broadband dB"); ax[1].set(xlabel="rpm", ylabel="dB", title="hover and axial cases at the listener"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
SOUND = pl.render(OUT / "sound", TAB, mass_kg=1.686)
for name, f in SOUND.items():
    extra = f" — {f['total_dba']:.0f} dB(A)" if "total_dba" in f else (f" — four rotors at {', '.join(f'{r:.0f}' for r in f['rpms'])} rpm, beating at {f['beat_hz'][0]:.1f} Hz" if "rpms" in f else "")
    print(f"{name}: {f['seconds']:.0f} s, {f['rms_db']:.0f} dB rms at the listener{extra} — {f['path']}")
    display(Audio(url=str(f["path"])))                     # linked, not embedded
print("the transition glide (embedded):"); display(Audio(str(SOUND["transition_hover_to_cruise"]["path"])))

# ---- the design dossier: every number of this notebook as Markdown to paste into a conversation (kestrel_report)
import kestrel_report as kr
def _g(n):
    return globals().get(n)
def _chk(names):
    import pandas as pd
    out = {}
    for n in names:
        c = _g(n)
        if c:
            out[n] = kl.check_table(c, n)
    return out

REPORT = {"settings": {"CFD": "on" if RUN_CFD else "NOT RUN", "quality": QUALITY, "mesh": pl.QUALITY[QUALITY], "kinds": ", ".join(ONLY)}, "the 9x6": G, "blade rows (Boreas)": BLADE.T,
          "rpm": {"hover": pl.rpm_hover(), "cruise": pl.rpm_cruise()}, "cases: CFD against BEMT": TAB[cols], "campaign": {k: v for k, v in SM.items() if k != "line"},
          "section fit": SEC, "normal-force fit": {k: v for k, v in NF.items() if k != "points"}, "normal-force points": NF["points"], "gates": ACC, "calibration on file": CAL, "noise at the listener": NZ,
          "sound files": pd.DataFrame(SOUND).T, "checks": kl.check_table(CHK, "the campaign")}
EXP = kr.export("38_prop_lab", "prop_lab — notebook 38, the 9x6 alone", REPORT, meta={"date": TODAY, "quality": QUALITY})
print("JSON report:", EXP["json"], f"({EXP['json_kB']} kB) | JSON dossier:", EXP["dossier_json"], f"({EXP['dossier_json_kB']} kB) | Markdown:", EXP["markdown"])
kr.paste("38_prop_lab", max_lines=30)
STAMPS = GATE.leave()                               # prop_lab.json and the report written by this run, stamped for 43_kestrel_study (docs/yanus.md)


## 7. Findings

**The first campaign (screening, 6 cells per diameter, 100 k cells, 400 iterations, none converged) solved all 21
cases and taught the notebook its own limits.** Thrust came out at **0.60 × BEMT** in the hover and at low advance
ratio with the torque at 0.92 → 0.65, the ratio collapsing with J (0.53 at J 0.52, 0.19 at 0.66, 0.28 at the cruise
point) and the hover's figure of merit at **0.29** against BEMT's 0.56 — a real 9×6 sits at 0.5–0.6. Four cells across
a 21 mm chord cannot carry a blade's camber and thickness: the CFD propeller behaved like one with less pitch and
less lift, which is what an under-resolved blade does, not what the propeller does. The two-parameter fit (cl × 0.58,
cd × 0.94) was poor (rms 0.37 of the log ratios after the fit) and the notebook nevertheless wrote it to the file and
the drives picked it up (the hover rpm moved 6723 → 7823). That is now impossible: a calibration is applied only when
it passes the gates (fit rms ≤ 0.15, scales within 0.7–1.3, the normal force from both blade azimuths), the loader
applies the same gates to an old file, and what fails is written down with its reason.

**The normal force came out at zero** (CNα −0.004 /rad against De Young's 0.32): the frozen rotor's blades lay in the
plane of the cross-flow, where the advancing and retreating sides are mirror images — a steady MRF at one azimuth
cannot see the normal force. The oblique cases now run at two blade azimuths (0° and 90°) and the normal force is
the mean; a transient sliding mesh is the step after that.

**What held**: the propeller windmills at 24 m/s in both (the pitch speed at 8000 rpm is 20.3 m/s), the four-rotor
hover beats at 3.9 Hz, every case ran to the end — the pipeline is sound, the mesh was not. Next run at the new
screening quality (10 cells per diameter, level 4, 600 iterations: ~0.5–1 M cells): read the thrust ratio against J
first — flat near 0.8–0.9 means a scaled generic section is the right model and the calibration goes through;
still falling with J means the blade's section itself is wrong, and the real APC profile on Zephyr is next.